In [1]:
import kuzu
from faker import Faker
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import random
from datetime import datetime
import uuid

ModuleNotFoundError: No module named 'kuzu'

In [ ]:
print("Opening Database")

db = kuzu.Database(r"../generated_files/instance_db")
conn = kuzu.Connection(db)

In [ ]:
start_baseline = np.datetime64('2008-01-01')

# Define means as dates
mean_dates = np.array(['2009-03-12', '2014-06-15', '2019-10-01', '2024-08-20'], dtype='datetime64[D]')

# Convert means and baseline to numerical days since the Unix epoch
baseline_days = start_baseline.astype(int)
means_days = mean_dates.astype(int)

stds = [365*2, 365*2, 365*2, 365*2] # Distribution
weights = [0.3, 0.25, 0.3, 0.15]
n_samples = 1000

choices = np.random.choice(len(weights), size=n_samples, p=weights)

samples = np.zeros(n_samples)
for i in range(len(weights)):
    indices = np.where(choices == i)[0]
    n_mode_samples = len(indices)

    # Draw initial samples for this mode
    mode_samples = np.random.normal(loc=means_days[i], scale=stds[i], size=n_mode_samples)

    # Rejection loop: replace any values <= 0
    while True:
        invalid = mode_samples <= baseline_days
        n_invalid = np.sum(invalid)
        if n_invalid == 0:
            break
        # Redraw only the invalid samples
        mode_samples[invalid] = np.random.normal(loc=means_days[i], scale=stds[i], size=n_invalid)

    samples[indices] = mode_samples

generated_dates = samples.astype('datetime64[D]')

# 4. Clean and Scannable Plot
fig, ax = plt.subplots(figsize=(10, 5))

# REDUCED BINS & REMOVED EDGECOLOR: Creates a smoother visual look
ax.hist(generated_dates, bins=50, alpha=0.7, color='seagreen', edgecolor='none')

# SPARSITY CONTROLS: Show labels every 2 months instead of every month
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=12))

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))

# Clean up axes styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3) # Subtle background grid

plt.title('Multi-modal Gaussian Distribution of Dates', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Count', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
industry_prefixes = {
    "Technology": 100, "Healthcare": 200, "Finance": 300,
    "Retail": 400, "Manufacturing": 500, "Real Estate": 600,
    "Entertainment": 700, "Education": 800, "Telecommunications": 900
}
industry_names = list(industry_prefixes.keys())

industry_size_weights = {
    "Technology": 1.3, "Healthcare": 1.4, "Finance": 1.5,
    "Retail": 1.0, "Manufacturing": 2, "Real Estate": 1.1,
    "Entertainment": 0.9, "Education": 0.7, "Telecommunications": 1.3
}

assigned_industries = []
assigned_sizes = []

era_2013 = np.datetime64('2013-01-01').astype(int)
era_2019 = np.datetime64('2019-01-01').astype(int)
era_2023 = np.datetime64('2023-01-01').astype(int)

for day in samples:
    if day < era_2013:
        probs = [0.05, 0.10, 0.35, 0.10, 0.25, 0.10, 0.02, 0.01, 0.02]
        era_size_modifier = 1.2
        target_industry_boost = "Finance"

    elif day < era_2019:
        probs = [0.40, 0.05, 0.05, 0.15, 0.05, 0.05, 0.15, 0.05, 0.05]
        era_size_modifier = 1.4
        target_industry_boost = "Technology"

    elif day < era_2023:
        probs = [0.15, 0.35, 0.05, 0.05, 0.05, 0.02, 0.10, 0.08, 0.15]
        era_size_modifier = 1.0
        target_industry_boost = "Healthcare"

    else:
        probs = [0.25, 0.15, 0.10, 0.15, 0.05, 0.05, 0.10, 0.10, 0.05]
        era_size_modifier = 0.6
        target_industry_boost = "Technology"

    ind = np.random.choice(industry_names, p=probs)
    assigned_industries.append(ind)

    base_industry_weight = industry_size_weights[ind]

    boost = 1.7 if ind == target_industry_boost else 1.0

    log_base = np.random.lognormal(mean=1.8, sigma=0.8)

    final_size = int(log_base * base_industry_weight * era_size_modifier * boost) + 5
    assigned_sizes.append(final_size)

assigned_industries = np.array(assigned_industries)
assigned_sizes = np.array(assigned_sizes)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

color_map = {
    "Technology": "#1f77b4", "Healthcare": "#2ca02c",
    "Finance": "#ff7f0e", "Manufacturing": "#d62728"
}

for ind in color_map.keys():
    mask = assigned_industries == ind
    ax1.scatter(generated_dates[mask], assigned_sizes[mask],
                color=color_map[ind], alpha=0.6, label=ind, s=25, edgecolors='none')

ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=24))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.set_yscale('log')
ax1.set_title('Company Size over Time (by Industry Type)', fontsize=13)
ax1.set_xlabel('Founding Date')
ax1.set_ylabel('Company Size (Log Scale)')
ax1.grid(True, which="both", linestyle='--', alpha=0.2)
ax1.legend()

for ind in color_map.keys():
    mask = assigned_industries == ind
    ax2.hist(generated_dates[mask], bins=25, alpha=0.4, label=ind, color=color_map[ind], edgecolor='none')

ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=24))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.set_title('Industry Type Distribution Over Time', fontsize=13)
ax2.set_xlabel('Founding Date')
ax2.set_ylabel('New Company Registrations Count')
ax2.grid(axis='y', linestyle='--', alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
cities = [
    "New York City", "Chicago", "San Francisco", "Austin", "Los Angeles",
    "Boston", "Seattle", "Houston", "Atlanta", "Miami"
]

states = {
    "New York City": "New York",
    "Chicago": "Illinois",
    "San Francisco": "California",
    "Austin": "Texas",
    "Los Angeles": "California",
    "Boston": "Massachusetts",
    "Seattle": "Washington",
    "Houston": "Texas",
    "Atlanta": "Georgia",
    "Miami": "Florida"
}

city_industry_weights = {
    "Technology": {"New York City": 0.15, "Chicago": 0.05, "San Francisco": 0.40, "Austin": 0.20, "Los Angeles": 0.10, "Boston": 0.05, "Seattle": 0.25, "Houston": 0.02, "Atlanta": 0.05, "Miami": 0.05},
    "Finance": {"New York City": 0.50, "Chicago": 0.25, "San Francisco": 0.10, "Austin": 0.05, "Los Angeles": 0.05, "Boston": 0.10, "Seattle": 0.02, "Houston": 0.03, "Atlanta": 0.05, "Miami": 0.15},
    "Healthcare": {"New York City": 0.10, "Chicago": 0.15, "San Francisco": 0.05, "Austin": 0.05, "Los Angeles": 0.10, "Boston": 0.35, "Seattle": 0.05, "Houston": 0.10, "Atlanta": 0.10, "Miami": 0.05},
    "Retail": {"New York City": 0.25, "Chicago": 0.15, "San Francisco": 0.05, "Austin": 0.10, "Los Angeles": 0.15, "Boston": 0.05, "Seattle": 0.15, "Houston": 0.10, "Atlanta": 0.15, "Miami": 0.10},
    "Manufacturing": {"New York City": 0.02, "Chicago": 0.30, "San Francisco": 0.02, "Austin": 0.05, "Los Angeles": 0.10, "Boston": 0.05, "Seattle": 0.10, "Houston": 0.25, "Atlanta": 0.10, "Miami": 0.02},
    "Real Estate": {"New York City": 0.30, "Chicago": 0.10, "San Francisco": 0.10, "Austin": 0.10, "Los Angeles": 0.20, "Boston": 0.05, "Seattle": 0.05, "Houston": 0.05, "Atlanta": 0.05, "Miami": 0.25},
    "Entertainment": {"New York City": 0.25, "Chicago": 0.05, "San Francisco": 0.05, "Austin": 0.05, "Los Angeles": 0.55, "Boston": 0.02, "Seattle": 0.02, "Houston": 0.02, "Atlanta": 0.10, "Miami": 0.15},
    "Education": {"New York City": 0.20, "Chicago": 0.15, "San Francisco": 0.10, "Austin": 0.10, "Los Angeles": 0.15, "Boston": 0.40, "Seattle": 0.05, "Houston": 0.05, "Atlanta": 0.10, "Miami": 0.10},
    "Telecommunications": {"New York City": 0.20, "Chicago": 0.15, "San Francisco": 0.10, "Austin": 0.15, "Los Angeles": 0.10, "Boston": 0.05, "Seattle": 0.05, "Houston": 0.10, "Atlanta": 0.25, "Miami": 0.05}
}

city_size_weights = {
    "New York City": 1.5, "Los Angeles": 1.4, "Chicago": 1.2,
    "San Francisco": 1.1, "Houston": 1.0, "Atlanta": 0.9,
    "Boston": 0.9, "Seattle": 0.9, "Austin": 0.8, "Miami": 0.8
}

assigned_cities = []

# 4. Loop through generated companies to assign a city based on combined probabilities
for ind, size in zip(assigned_industries, assigned_sizes):
    probs = []

    for city in cities:
        # Fetch the industry preference for this city
        ind_weight = city_industry_weights[ind][city]

        # Fetch the size preference for this city
        size_weight = city_size_weights[city]

        # Scaling factor: The larger the company, the heavier the city's size weight matters
        # Using a log scale on size ensures massive sizes don't completely break the distribution
        size_effect = np.log1p(size) * size_weight

        # Combine the industry preference and the size effect
        combined_score = ind_weight * size_effect
        probs.append(combined_score)

    # Normalize scores into a valid probability distribution (sum to 1.0)
    probs = np.array(probs)
    probs /= probs.sum()

    # Randomly select a city using the unique probabilities of this company
    chosen_city = np.random.choice(cities, p=probs)
    assigned_cities.append(chosen_city)

assigned_cities = np.array(assigned_cities)

In [ ]:

df = pd.DataFrame({
    "Date": generated_dates,
    "Industry": assigned_industries,
    "Size": assigned_sizes,
    "City": assigned_cities
})

# 2. Set up a 2-panel visualization dashboard
fig, axes = plt.subplots(2, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [1, 1.2]})
sns.set_theme(style="whitegrid")

# PANEL A: Industry Distribution by City (Verification of Type Clumping)
# Create a cross-tabulation table normalized by row to display percentages
city_industry_pct = pd.crosstab(df['City'], df['Industry'], normalize='index') * 100

city_industry_pct.plot(
    kind='bar',
    stacked=True,
    ax=axes[0],
    colormap='tab20'
)
axes[0].set_title("Panel A: Industry Composition Breakdown per City (%)", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Percentage (%)")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title="Industries", bbox_to_anchor=(1.02, 1), loc='upper left')

# PANEL B: Company Size Distribution by City & Industry (Verification of Size + Type Clumping)
# Filter for key illustrative industries to keep the plot highly scannable
target_industries = ["Technology", "Finance", "Healthcare", "Entertainment"]
df_filtered = df[df['Industry'].isin(target_industries)]

sns.stripplot(
    data=df_filtered,
    x="City",
    y="Size",
    hue="Industry",
    jitter=0.25,
    size=6,
    alpha=0.75,
    palette="Set1",
    dodge=True,
    ax=axes[1]
)
axes[1].set_title("Panel B: Company Size Distribution Across Cities for Key Sectors", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Company Size Value")
axes[1].set_xlabel("Assigned City Hub")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
locales = {
    'en_US': 0.7,
    'en_GB': 0.2,
    'en_IN': 0.1
}

fake = Faker(locales)

def random_gaussian_in_range(mean, std, low, high):
    while True:
        sample = random.gauss(mean, std)
        if low <= sample <= high:
            return sample

categories = ["American", "Indian", "British", "Mexican"]

probabilities = [0.85, 0.03, 0.02, 0.10]

In [ ]:
from df_types import *

# Initialize empty lists for Nodes
nodes_company = []
nodes_person = []
nodes_address = []
nodes_device = []
nodes_account = []
nodes_document = []

# Initialize empty lists for Relationships
rel_works_for = []
rel_directs = []
rel_company_has_address = []
rel_person_has_address = []
rel_company_uses_device = []
rel_person_uses_device = []
rel_company_owns_account = []
rel_person_owns_account = []
rel_company_provided_doc = []
rel_person_provided_doc = []
rel_person_owns_equity = [] # <--- NEW EQUITY LIST

# (Placeholders for later generation)
# rel_matches_watchlist = []
# rel_sent_tx = []
# rel_recieved_tx = []

# Helper lists for random generation
device_types = ["Mobile", "Desktop", "Tablet"]
os_types = ["iOS", "Android", "Windows", "macOS", "Linux"]
account_types = ["Checking", "Savings", "Corporate"]
doc_types_person = ["Passport", "Driver License", "National ID"]
doc_types_company = ["Articles of Incorporation", "Tax Certificate", "Business License"]

# Baseline year to calculate company age
CURRENT_YEAR = 2026

In [ ]:
# Start Generation Loop
# --- PRE-GENERATE VIRTUAL OFFICES ---
# Create 5 classic "Virtual Office" hubs (e.g., in Delaware or Wyoming)
num_virtual_offices = 5
virtual_offices = []
for v in range(num_virtual_offices):
    v_id = f"addr_vo_{v}"
    virtual_offices.append({
        "addressId": v_id,
        "street": fake.street_address() + f" Suite {random.randint(100,999)}",
        "city": "Wilmington",
        "state": "Delaware",
        "zipCode": fake.zipcode(),
        "country": "USA"
    })
    nodes_address.append(virtual_offices[-1]) # Add them to the global nodes list

# --- PRE-GENERATE PROFESSIONAL NOMINEES ---
num_nominees = 3
nominee_ids = []
for n in range(num_nominees):
    n_id = f"nominee_{n}"
    nominee_ids.append(n_id)
    nodes_person.append({
        "personId": n_id,
        "firstName": fake.first_name(),
        "lastName": fake.last_name(),
        "dob": datetime(1970 + n, 1, 1),
        "taxId": f"NOM_{n}",
        "nationality": "American",
        "pepStatus": False,
        "kycStatus": "True",
        "riskScore": 95.0 # Nominees are inherently high risk
    })


for i in tqdm(range(len(df)), desc="Generating Companies and Data", colour="red"):
    c_id = str(i)
    inc_date = df["Date"].iloc[i]
    ind_code = str(industry_prefixes[df["Industry"].iloc[i]])
    is_orphan_company = random.random() < 0.05

    age_in_years = max(0, CURRENT_YEAR - inc_date.year)
    base_equity_chance = min(0.10, 0.01 + (age_in_years * 0.005))

    nodes_company.append({
        "companyId": c_id,
        "name": fake.unique.company(),
        "regNumber": f"r{ind_code}{i}",
        "incorporationDate": inc_date,
        "industryCode": ind_code,
        "kycStatus": str(random.random() < 0.9),
        "riskScore": random_gaussian_in_range(35, 50, 0, 100)
    })

    if random.random() < 0.05:
        # 5% chance to use a shared Virtual Office
        c_addr_id = random.choice(virtual_offices)["addressId"]
    else:
        # Normal unique address
        c_addr_id = f"addr_c_{i}"
        nodes_address.append({
            "addressId": c_addr_id,
            "street": fake.street_address(),
            "city": df["City"].iloc[i],
            "state": states[df["City"].iloc[i]],
            "zipCode": fake.zipcode(),
            "country": "USA"
        })

    rel_company_has_address.append({"_from": c_id, "_to": c_addr_id, "addressType": "Business", "isCurrent": True})

    c_dev_id = f"dev_c_{i}"
    nodes_device.append({
        "deviceId": c_dev_id,
        "deviceType": random.choice(device_types),
        "os": random.choice(os_types),
        "ipAddress": fake.ipv4(),
        "macAddress": fake.mac_address(),
        "isp": fake.company()
    })
    rel_company_uses_device.append({"_from": c_id, "_to": c_dev_id, "firstSeen": inc_date, "lastSeen": inc_date, "trustScore": 90.0})

    # 4. Company Account (and 14. Document)
    c_acc_id = f"acc_c_{i}"
    nodes_account.append({
        "accountId": c_acc_id,
        "accountType": "Corporate",
        "balance": round(random.uniform(5000, 500000), 2),
        "currency": "USD",
        "status": "Active",
        "openedDate": inc_date,
        "branchCode": str(random.randint(1000, 9999))
    })
    rel_company_owns_account.append({"_from": c_id, "_to": c_acc_id, "role": "Primary Entity", "since": inc_date})

    c_doc_id = f"doc_c_{i}"
    nodes_document.append({
        "docId": c_doc_id,
        "docType": random.choice(doc_types_company),
        "issuedCountry": "USA",
        "expiryDate": inc_date + pd.Timedelta(days=365*5),
        "isForged": bool(random.random() < 0.01)
    })
    rel_company_provided_doc.append({"_from": c_id, "_to": c_doc_id, "submissionDate": inc_date, "verificationMethod": "Automated"})

    for j in range(df["Size"].iloc[i]):
        p_id = f"{i}W{j}R0"
        last_name = fake.last_name()
        dob = datetime(inc_date.year - np.random.randint(22, 56), np.random.randint(1, 13), np.random.randint(1, 28))

        nodes_person.append({
            "personId": p_id,
            "firstName": fake.first_name(),
            "lastName": last_name,
            "dob": dob,
            "taxId": f"{p_id}I{ind_code}",
            "nationality": random.choices(categories, weights=probabilities, k=1)[0],
            "pepStatus": bool(random.random() < 0.07),
            "kycStatus": str(random.random() < 0.9),
            "riskScore": random_gaussian_in_range(45, 35, 0, 100)
        })

        rel_works_for.append({
            "_from": p_id, "_to": c_id,
            "jobTitle": fake.job(),
            "employmentType": random.choice(["Full-time", "Contractor"]),
            "startDate": inc_date,
            "salaryRange": str(random.randint(40000, 150000))
        })

        # 12. Directs Company Relation (Nominee Typology)
        if random.random() < 0.01:
            # 1% chance a professional nominee directs this company (Massive Star Graph)
            nominee = random.choice(nominee_ids)
            rel_directs.append({"_from": nominee, "_to": c_id, "role": "Director", "appointedDate": inc_date})
        elif random.random() < 0.005:
            # 0.5% chance the primary employee directs it
            rel_directs.append({"_from": p_id, "_to": c_id, "role": "Director", "appointedDate": inc_date})

        # 13. Primary Person Owns Equity (UBO Evasion Logic)
        primary_owns_equity = False
        if random.random() < base_equity_chance:
            primary_owns_equity = True

            # If orphan, cap at 24%. Otherwise, they can own up to 100%.
            max_equity = 24.0 if is_orphan_company else 100.0
            pct = round(random.uniform(0.5, max_equity), 2)

            rel_person_owns_equity.append({
                "_from": p_id,
                "_to": c_id,
                "percentage": pct,
                "votingRights": pct if random.random() < 0.8 else 0.0
            })

        p_addr_id = f"addr_p_{i}_{j}"
        nodes_address.append({
            "addressId": p_addr_id,
            "street": fake.street_address(),
            "city": df["City"].iloc[i],
            "state": states[df["City"].iloc[i]],
            "zipCode": fake.zipcode(),
            "country": "USA"
        })
        rel_person_has_address.append({"_from": p_id, "_to": p_addr_id, "addressType": "Residential", "isCurrent": True})

        p_dev_id = f"dev_p_{i}_{j}"

        primary_os = random.choices(os_types, weights=[0.28, 0.17, 0.25, 0.20, 0.10], k=1)[0]

        is_high_risk = nodes_person[-1]["riskScore"] > 75.0
        mismatch_chance = 0.60 if is_high_risk else 0.05

        device_country = "USA" # Default matches physical address
        if random.random() < mismatch_chance:
            device_country = fake.country() # Mismatch! Simulates VPN / Offshore

        nodes_device.append({
            "deviceId": p_dev_id,
            "deviceType": random.choice(["Mobile", "Desktop"]),
            "os": primary_os,
            "ipAddress": fake.ipv4(),
            "macAddress": fake.mac_address(),
            "isp": fake.company(),
            "registeredCountry": device_country # <-- Required for Kùzu querying
        })
        rel_person_uses_device.append({"_from": p_id, "_to": p_dev_id, "firstSeen": inc_date, "lastSeen": inc_date, "trustScore": 85.0})

        p_acc_id = f"acc_p_{i}_{j}"
        nodes_account.append({
            "accountId": p_acc_id,
            "accountType": random.choice(["Checking", "Savings"]),
            "balance": round(random.uniform(100, 25000), 2),
            "currency": "USD",
            "status": "Active",
            "openedDate": inc_date,
            "branchCode": str(random.randint(1000, 9999))
        })
        rel_person_owns_account.append({"_from": p_id, "_to": p_acc_id, "role": "Owner", "since": inc_date})

        p_doc_id = f"doc_p_{i}_{j}"
        nodes_document.append({
            "docId": p_doc_id,
            "docType": random.choice(doc_types_person),
            "issuedCountry": "USA",
            "expiryDate": inc_date + pd.Timedelta(days=365*4),
            "isForged": bool(random.random() < 0.05)
        })
        rel_person_provided_doc.append({"_from": p_id, "_to": p_doc_id, "submissionDate": inc_date, "verificationMethod": "Manual"})

        num_relatives = np.random.randint(0, 4)
        for k in range(1, num_relatives + 1):
            rp_id = f"{i}W{j}R{k}" # R1, R2, R3 denote relatives

            nodes_person.append({
                "personId": rp_id,
                "firstName": fake.first_name(),
                "lastName": last_name, # Usually share last name
                "dob": datetime(inc_date.year - np.random.randint(18, 70), np.random.randint(1, 13), np.random.randint(1, 28)),
                "taxId": f"{rp_id}I{ind_code}",
                "nationality": random.choices(categories, weights=probabilities, k=1)[0],
                "pepStatus": bool(random.random() < 0.07),
                "kycStatus": str(random.random() < 0.9),
                "riskScore": random_gaussian_in_range(45, 35, 0, 100)
            })

            rel_equity_chance = 0.40 if primary_owns_equity else base_equity_chance
            if random.random() < rel_equity_chance:
                pct = round(random.uniform(0.1, 15.0), 2)
                rel_person_owns_equity.append({
                    "_from": rp_id,
                    "_to": c_id,
                    "percentage": pct,
                    "votingRights": pct if random.random() < 0.8 else 0.0
                })

            rel_person_has_address.append({"_from": rp_id, "_to": p_addr_id, "addressType": "Residential", "isCurrent": True})

            if random.random() < 0.40:
                rel_person_uses_device.append({"_from": rp_id, "_to": p_dev_id, "firstSeen": inc_date, "lastSeen": inc_date, "trustScore": 75.0})
            else:
                # 60% chance they have a unique device, heavily influenced by primary OS
                if primary_os == "iOS" or primary_os == "Mac":
                    rel_os = random.choices(os_types, weights=[0.40, 0.10, 0.10, 0.30, 0.10], k=1)[0]
                else:
                    rel_os = random.choices(os_types, weights=[0.28, 0.17, 0.25, 0.20, 0.10], k=1)[0]

                # --- NEW: IP Mismatch Logic for Relative ---
                is_rp_high_risk = nodes_person[-1]["riskScore"] > 75.0
                rp_mismatch_chance = 0.60 if is_rp_high_risk else 0.05

                rp_device_country = "USA"
                if random.random() < rp_mismatch_chance:
                    rp_device_country = fake.country()

                rp_dev_id = f"dev_rp_{i}_{j}_{k}"
                nodes_device.append({
                    "deviceId": rp_dev_id,
                    "deviceType": random.choice(["Mobile", "Tablet"]),
                    "os": rel_os,
                    "ipAddress": fake.ipv4(),
                    "macAddress": fake.mac_address(),
                    "isp": fake.company(),
                    "registeredCountry": rp_device_country # <-- Required here too
                })
                rel_person_uses_device.append({"_from": rp_id, "_to": rp_dev_id, "firstSeen": inc_date, "lastSeen": inc_date, "trustScore": 80.0})
                
            rp_acc_id = f"acc_rp_{i}_{j}_{k}"
            nodes_account.append({
                "accountId": rp_acc_id,
                "accountType": random.choice(["Checking", "Savings"]),
                "balance": round(random.uniform(50, 10000), 2),
                "currency": "USD",
                "status": "Active",
                "openedDate": inc_date,
                "branchCode": str(random.randint(1000, 9999))
            })
            rel_person_owns_account.append({"_from": rp_id, "_to": rp_acc_id, "role": "Owner", "since": inc_date})

            rp_doc_id = f"doc_rp_{i}_{j}_{k}"
            nodes_document.append({
                "docId": rp_doc_id,
                "docType": random.choice(doc_types_person),
                "issuedCountry": "USA",
                "expiryDate": inc_date + pd.Timedelta(days=365*4),
                "isForged": bool(random.random() < 0.05)
            })
            rel_person_provided_doc.append({"_from": rp_id, "_to": rp_doc_id, "submissionDate": inc_date, "verificationMethod": "Manual"})

print("Converting generated data to DataFrames...")

# Convert Nodes
df_Company = pd.DataFrame(nodes_company).astype(dtype_Company)
df_Person = pd.DataFrame(nodes_person).astype(dtype_Person)
df_Address = pd.DataFrame(nodes_address).astype(dtype_Address)
df_Device = pd.DataFrame(nodes_device).astype(dtype_Device)
df_Account = pd.DataFrame(nodes_account).astype(dtype_Account)
df_Document = pd.DataFrame(nodes_document).astype(dtype_Document)

# Convert Relationships
df_WORKS_FOR = pd.DataFrame(rel_works_for).astype(dtype_WORKS_FOR)
df_DIRECTS = pd.DataFrame(rel_directs).astype(dtype_DIRECTS)

df_COMPANY_HAS_ADDRESS = pd.DataFrame(rel_company_has_address).astype(dtype_HAS_ADDRESS)
df_PERSON_HAS_ADDRESS = pd.DataFrame(rel_person_has_address).astype(dtype_HAS_ADDRESS)

df_COMPANY_USES_DEVICE = pd.DataFrame(rel_company_uses_device).astype(dtype_USES_DEVICE)
df_PERSON_USES_DEVICE = pd.DataFrame(rel_person_uses_device).astype(dtype_USES_DEVICE)

df_COMPANY_OWNS_ACCOUNT = pd.DataFrame(rel_company_owns_account).astype(dtype_OWNS_ACCOUNT)
df_PERSON_OWNS_ACCOUNT = pd.DataFrame(rel_person_owns_account).astype(dtype_OWNS_ACCOUNT)

df_COMPANY_PROVIDED_DOC = pd.DataFrame(rel_company_provided_doc).astype(dtype_PROVIDED_DOC)
df_PERSON_PROVIDED_DOC = pd.DataFrame(rel_person_provided_doc).astype(dtype_PROVIDED_DOC)

# NEW: Convert Equity DataFrame
df_PERSON_OWNS_EQUITY = pd.DataFrame(rel_person_owns_equity).astype(dtype_OWNS_EQUITY)

print("Data generation complete!")

In [ ]:
save_all_dfs_to_csv(r"../generated_files/")